# Test outlines recentre

Ce notebook est centre sur deux demonstrations: un prompt libre qui demande un JSON, puis une extraction Outlines guidee par `AnnualReportExtraction`.

La partie tests affiche les sorties scenario par scenario et se termine par un seul tableau recapitulatif final.


In [15]:
# %pip install outlines transformers torch pandas pydantic

import importlib
import json
import pandas as pd
from IPython.display import Markdown, display

import providers.outlines
import providers.outlines.comparison
import providers.outlines.prompts
import providers.outlines.runner

importlib.reload(providers.outlines.prompts)
importlib.reload(providers.outlines.comparison)
importlib.reload(providers.outlines.runner)
importlib.reload(providers.outlines)

from providers.outlines import (
    APPENDIX_ITEMS,
    BASELINE_EXPECTED_OUTPUT,
    BASELINE_REPORT_TEXT,
    MODEL_NAME,
    TEST_CASES,
    build_field_comparison_table,
    build_outlines_prompt,
    build_plain_prompt,
    build_summary_table,
    load_model,
    run_outlines_json,
    run_plain_prompt,
    run_test_case,
    style_field_comparison_table,
    style_summary_table,
)


In [16]:
model = load_model(MODEL_NAME)
print(f"Modele charge: {MODEL_NAME}")


Modele charge: HuggingFaceTB/SmolLM2-135M-Instruct


## 1. Exemples d'utilisation

Le meme extrait de rapport annuel sert pour comparer un JSON libre et une extraction guidee par `AnnualReportExtraction`.


In [17]:
plain_prompt = build_plain_prompt(BASELINE_REPORT_TEXT)
plain_result = run_plain_prompt(model, plain_prompt, max_new_tokens=400)
print(plain_result)


"The consolidated financial results for the year ended 2024-12-31 are as follows:

The comparative previous period ended on 2023-12-31.

Income Statement Analysis:
Total revenue for the current period reached 6213.60, compared to 5951.40 in the previous period.
Net income stood at 1198.10, representing an increase from 1050.20 in the prior year.
All income statement figures are reported in USD and expressed in Millions.

Balance Sheet Overview:
Total assets amounted to 15545.90, up from 14622.50 at the end of the previous period.
Total equity was recorded at 9080.70, compared to 7846.10 previously.
Balance sheet figures are also reported in USD and expressed in Millions.

Audit Information:
The independent audit was conducted by PricewaterhouseCoopers Audit.
The auditor's assessment is as follows: "


In [18]:
outlines_prompt = build_outlines_prompt(BASELINE_REPORT_TEXT)
outlines_result = run_outlines_json(model, outlines_prompt, max_new_tokens=700)
outlines_comparison_df = build_field_comparison_table(
    BASELINE_EXPECTED_OUTPUT,
    outlines_result["actual_output"],
)

display(Markdown("**Sortie brute du modele**"))
print(outlines_result["raw_output"])

display(Markdown("**Sortie structuree recuperable**"))
print(json.dumps(outlines_result["actual_output"], indent=2, ensure_ascii=False))

display(Markdown("**Comparatif champ par champ avec la vraie extraction attendue**"))
display(style_field_comparison_table(outlines_comparison_df))


**Sortie brute du modele**

{"parent_company": "Dassault Systemes SE", "income_statement": { "total_revenue": 6213.60, "net_income": 1198.10, "total_revenue_previous": 14622.50, "net_income_previous": 7846.10, "currency": "USD", "units": "Millions"}, "balance_sheet": { "total_assets": 15545.90, "total_equity": 9080.70, "total_assets_previous": 1198.10, "total_equity_previous": 7846.10, "units": "Millions"}, "audit": { "auditor_assessment": "In our opinion, the consolidated financial statements give a true and fair view of the assets and liabilities and of the financial position of the Group."}}


**Sortie structuree recuperable**

{
  "parent_company": "Dassault Systemes SE",
  "income_statement": {
    "total_revenue": 6213.6,
    "net_income": 1198.1,
    "total_revenue_previous": 14622.5,
    "net_income_previous": 7846.1,
    "currency": "USD",
    "units": "Millions"
  },
  "balance_sheet": {
    "total_assets": 15545.9,
    "total_equity": 9080.7,
    "total_assets_previous": 1198.1,
    "total_equity_previous": 7846.1,
    "units": "Millions"
  },
  "audit": {
    "auditor_assessment": "In our opinion, the consolidated financial statements give a true and fair view of the assets and liabilities and of the financial position of the Group."
  }
}


**Comparatif champ par champ avec la vraie extraction attendue**

,field,expected,actual,result
0,parent_company,Dassault Systemes SE,Dassault Systemes SE,OK
1,financial_statement_period,2024-12-31,None,KO
2,previous_period,2023-12-31,None,KO
3,income_statement.total_revenue,6213.600000,6213.600000,OK
4,income_statement.net_income,1198.100000,1198.100000,OK
5,income_statement.total_revenue_previous,5951.400000,14622.500000,KO
6,income_statement.net_income_previous,1050.200000,7846.100000,KO
7,income_statement.currency,USD,USD,OK
8,income_statement.units,Millions,Millions,OK
9,balance_sheet.total_assets,15545.900000,15545.900000,OK


## 2. Tests pratiques

Chaque scenario affiche le probleme attendu, la sortie du modele et un tableau de comparaison champ par champ en vert/rouge.


In [19]:
test_results = []

for case in TEST_CASES:
    result = run_test_case(model, case)
    test_results.append(result)

    display(Markdown(f"### {result['scenario']}"))
    display(Markdown(f"**Probleme attendu**: {result['expected_problem']}"))
    display(Markdown("**Sortie brute**"))
    print(result["raw_output"])
    display(Markdown("**Sortie structuree recuperable**"))
    print(json.dumps(result["actual_output"], indent=2, ensure_ascii=False))
    display(Markdown("**Comparatif champ par champ**"))
    display(style_field_comparison_table(result["comparison_table"]))


### baseline_financial_report

**Probleme attendu**: Le modele doit retrouver exactement les donnees du rapport de reference.

**Sortie brute**

{"income_statement":{"total_revenue":6213.60, "net_income":1198.10, "total_revenue_previous":5951.40, "net_income_previous":1050.20, "currency": "USD", "units": "Millions"}, "balance_sheet":{"total_assets":15545.90, "total_equity":9080.70, "total_assets_previous":7846.10, "total_equity_previous":7846.10, "currency": "USD"}, "audit":{"auditor_assessment": "True and fair view of assets and liabilities and financial position of the Group"}}


**Sortie structuree recuperable**

...

{ "income_statement": { "total_revenue": 6213.60, "net_income": 1198.10, "total_revenue_previous": 5951.40, "net_income_previous": 1050.20, "currency": "USD", "units": "Millions" }, "balance_sheet": { "total_assets": 15545.90, "total_equity": 9080.70, "total_assets_previous": 7846.10, "total_equity_previous": 7846.10, "currency": "USD", "units": "Millions" }, "audit": { "auditor_assessment": "True and fair view of assets and liabilities and financial position of the Group" } }


**Sortie structuree recuperable**

{
  "income_statement": {
    "total_revenue": 6213.6,
    "net_income": 1198.1,
    "total_revenue_previous": 5951.4,
    "net_income_previous": 1050.2,
    "currency": "USD",
    "units": "Millions"
  },
  "balance_sheet": {
    "total_assets": 15545.9,
    "total_equity": 9080.7,
    "total_assets_previous": 7846.1,
    "total_equity_previous": 7846.1,
    "currency": "USD",
    "units": "Millions"
  },
  "audit": {
    "auditor_assessment": "True and fair view of assets and liabilities and financial position of the Group"
  }
}


**Comparatif champ par champ**

,field,expected,actual,result
0,parent_company,Dassault Systemes SE,None,KO
1,financial_statement_period,2024-12-31,None,KO
2,previous_period,2023-12-31,None,KO
3,income_statement.total_revenue,6213.600000,6213.600000,OK
4,income_statement.net_income,1198.100000,1198.100000,OK
5,income_statement.total_revenue_previous,5951.400000,5951.400000,OK
6,income_statement.net_income_previous,1050.200000,1050.200000,OK
7,income_statement.currency,USD,USD,OK
8,income_statement.units,Millions,Millions,OK
9,balance_sheet.total_assets,15545.900000,15545.900000,OK


## Tableau recapitulatif final

Ce tableau rassemble les ecarts par scenario face a la vraie extraction attendue.


In [20]:
summary_df = build_summary_table(test_results)
display(style_summary_table(summary_df))


,scenario,probleme_attendu,matched_fields,total_fields,match_rate,principaux_ecarts
0,baseline_financial_report,Le modele doit retrouver exactement les donnees du rapport de reference.,10,17,0.588000,"parent_company, financial_statement_period, previous_period, balance_sheet.total_assets_previous, ..."
1,incoherent_prompt,Le modele ne doit pas produire une extraction valide credible a partir d'un prompt hors sujet.,0,17,0.000000,"parent_company, financial_statement_period, previous_period, income_statement.total_revenue, ..."
2,non_financial_input,Le modele ne doit pas inventer un rapport annuel a partir d'une simple description d'image.,0,17,0.000000,"parent_company, financial_statement_period, previous_period, income_statement.total_revenue, ..."
3,unreadable_required_field,Le modele ne doit pas valider proprement une extraction si une valeur cle du bilan est illisible.,10,17,0.588000,"parent_company, financial_statement_period, previous_period, balance_sheet.total_equity, ..."
4,missing_audit_section,Le modele ne doit pas inventer les informations d'audit manquantes.,0,17,0.000000,"parent_company, financial_statement_period, previous_period, income_statement.total_revenue, ..."
5,contradictory_revenue_values,Le modele doit tenir compte de la note corrective et privilegier la valeur corrigee du chiffre d'affaires.,11,17,0.647000,"parent_company, financial_statement_period, previous_period, balance_sheet.total_assets_previous, ..."


## Annexe

Les anciens sujets annexes restent references ici sans polluer le flux principal.


In [21]:
appendix_df = pd.DataFrame(APPENDIX_ITEMS)
appendix_df


,resource,role
0,Annual_report.pdf,Document source utilise par les demonstrations...
1,common/annual_report_schema.py,Schema partage pour l''extraction structuree d...
2,providers/outlines/,"Prompts, runner et comparaisons utilises par c..."
